In [ ]:
import json
import os
import pandas as pd
import shutil
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 1. CLEANUP & SETUP
# ==========================================
WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "food_samples_final")
ZIP_NAME = os.path.join(WORKING_DIR, "food_samples_package")
EXCEL_PATH = os.path.join(WORKING_DIR, "food_class_frequency.xlsx")

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for f in [f"{ZIP_NAME}.zip", EXCEL_PATH]:
    if os.path.exists(f): os.remove(f)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. SMART PATH DETECTION
# ==========================================
BASE_PATH = "/kaggle/input/datasets/sainikhileshreddy/food-recognition-2022"
ANN_FILE = None
for p in Path(BASE_PATH).rglob("annotations.json"):
    if "public_training_set" in str(p):
        ANN_FILE = str(p)
        break

if not ANN_FILE:
    for p in Path(BASE_PATH).rglob("annotations.json"):
        ANN_FILE = str(p)
        break

IMG_ROOT = os.path.join(os.path.dirname(ANN_FILE), "images")

print(f"✅ Target Annotations: {ANN_FILE}")
print(f"✅ Target Images Folder: {IMG_ROOT}")

if not os.path.exists(IMG_ROOT) or len(os.listdir(IMG_ROOT)) == 0:
    print("⚠️ Warning: Images folder looks empty, searching for any 'images' folder...")
    for p in Path(BASE_PATH).rglob("images"):
        if len(os.listdir(str(p))) > 0:
            IMG_ROOT = str(p)
            print(f"✅ Found working images folder at: {IMG_ROOT}")
            break

# ==========================================
# 3. DATA PROCESSING
# ==========================================
with open(ANN_FILE, 'r') as f:
    data = json.load(f)

image_map = {img['id']: img['file_name'] for img in data['images']}
class_frequency = {}
samples_collector = {}

for ann in data['annotations']:
    cat_id = ann['category_id']
    img_id = ann['image_id']
    class_frequency[cat_id] = class_frequency.get(cat_id, 0) + 1
    if cat_id not in samples_collector: samples_collector[cat_id] = set()
    if len(samples_collector[cat_id]) < 5:
        fname = image_map.get(img_id)
        if fname: samples_collector[cat_id].add(fname)

# ==========================================
# 4. COPYING & VERIFICATION
# ==========================================
excel_rows = []
total_copied = 0

for cat in tqdm(data['categories'], desc="Copying Samples"):
    cat_id, cat_name = cat['id'], cat['name']
    safe_folder = os.path.join(OUTPUT_DIR, cat_name.replace("/", "-"))
    os.makedirs(safe_folder, exist_ok=True)
    
    actual_copied = 0
    for fname in samples_collector.get(cat_id, []):
        src = os.path.join(IMG_ROOT, fname)
        if os.path.exists(src):
            shutil.copy(src, safe_folder)
            actual_copied += 1
            total_copied += 1
            
    excel_rows.append({
        "Class_ID": cat_id,
        "Food_Name": cat_name,
        "Total_Frequency": class_frequency.get(cat_id, 0),
        "Images_Copied": actual_copied
    })

# ==========================================
# 5. EXCEL & ZIP
# ==========================================
pd.DataFrame(excel_rows).to_excel(EXCEL_PATH, index=False)
if total_copied > 0:
    print(f"Success! Copied {total_copied} images.")
    shutil.make_archive(ZIP_NAME, 'zip', OUTPUT_DIR)
    print(f"ZIP and Excel are ready in the output sidebar.")
else:
    print("ERROR: Still no images found. Check if the dataset is private or corrupted.")

✅ Target Annotations: /kaggle/input/datasets/sainikhileshreddy/food-recognition-2022/raw_data/public_training_set_release_2.0/annotations.json
✅ Target Images Folder: /kaggle/input/datasets/sainikhileshreddy/food-recognition-2022/raw_data/public_training_set_release_2.0/images


Copying Samples: 100%|██████████| 498/498 [00:09<00:00, 52.35it/s]


🔥 Success! Copied 2490 images.
✅ ZIP and Excel are ready in the output sidebar.
